# 15 CFM批量谱图整合与Sim-to-Real审计

**目的**：严格执行14号冻结的5,822个CFM-ID命令，标准化17,466张10/20/40 eV正负模式模拟谱，
并在预先冻结的公共Development实测对照上量化模拟域与真实域的差距。

本Notebook回答三个问题：

1. CFM大规模生成是否稳定，失败集中在哪些结构？
2. 生成谱是否满足基本物理和数据格式要求？
3. CFM谱应与真实谱同权重训练、低权重增强，还是只用于预训练？

**严格边界**：不读取ChemVista谱图或盲测性能；不新增14号之外的结构；不修改Novel排除阈值；
CFM谱只允许进入训练侧，绝不进入任何外折测试或ChemVista测试。

## 运行说明

- 默认 `RUN_BATCH=True`，从头运行会开始5,822个任务，预计需要数小时。
- 保持Docker Desktop运行，电脑不要休眠。
- 状态每10个任务原子保存一次；意外中断后重新从头运行Notebook即可断点续跑。
- 已完整生成的原始文件不会重复计算。
- 已失败任务默认不反复重试；明确修复原因后才把 `RERUN_FAILED=True`。
- 正式运行必须保持 `MAX_JOBS=None`。该变量仅用于代码调试，截断运行不能形成正式结果。

生成完成后还会读取约3,192个冻结的公共实测对照，计算modified cosine、谱图熵相似度、峰召回率、
匹配峰强度相关性、峰数/质量分布和real-vs-synthetic域分类器。

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import pickle
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm
from scipy import sparse
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupKFold
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 240)
sns.set_theme(style='whitegrid', context='notebook')

# -------------------- 用户运行配置 --------------------
RUN_BATCH = False
RERUN_FAILED = False
MAX_JOBS = None
COMMAND_TIMEOUT_SECONDS = 30 * 60
SAVE_STATUS_EVERY = 10
# ----------------------------------------------------

# -------------------- 冻结分析参数 --------------------
COLLISION_ENERGY_MAP = {0: 10, 1: 20, 2: 40}
MZ_TOLERANCE_DA = 0.02
PRECURSOR_MZ_TOLERANCE_DA = 0.01
MAX_REAL_CE_DELTA_EV = 10.0
DOMAIN_BIN_WIDTH_DA = 1.0
DOMAIN_MAX_MZ = 2000.0
DOMAIN_CV_SPLITS = 5
SEED = 20260905
MIN_OVERALL_GENERATION_SUCCESS = 0.95
MIN_MODE_GENERATION_SUCCESS = 0.90
# ----------------------------------------------------

cwd = Path.cwd().resolve()
roots = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((path for path in roots if (path / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请手动设置 ROOT。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
PLAN_DIR = BASE / 'cfm_v1' / 'frozen_generation_plan_v1'
OUTPUT_DIR = BASE / 'cfm_v1' / 'batch_spectra_v1'
RAW_DIR = OUTPUT_DIR / 'raw'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for path in (OUTPUT_DIR, RAW_DIR / 'positive', RAW_DIR / 'negative', FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

PATHS = {
    'plan_manifest': PLAN_DIR / 'cfm_generation_plan_manifest.json',
    'generation_jobs': PLAN_DIR / 'cfm_generation_jobs.csv',
    'compound_manifest': PLAN_DIR / 'cfm_compound_manifest.csv',
    'fold_permissions': PLAN_DIR / 'cfm_outer_fold_permission.csv',
    'reference_pairs': PLAN_DIR / 'cfm_sim_to_real_reference_pairs.csv',
    'plan_qc': PLAN_DIR / 'cfm_generation_plan_qc.csv',
    'public_manifest': BASE / 'spectrum_audit_v1' / 'public_relevant_spectrum_manifest.csv.gz',
    'public_stream': ROOT / '质谱数据' / 'EnvMolBench_Cleaned.pkl',
}
missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少15号输入文件：\n' + '\n'.join(missing)

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Python:', sys.executable)
print('RDKit:', rdBase.rdkitVersion)
print('RUN_BATCH:', RUN_BATCH, 'RERUN_FAILED:', RERUN_FAILED, 'MAX_JOBS:', MAX_JOBS)

## 1. 验证14号冻结计划

正式运行前同时检查14号状态、全部QC、任务文件哈希、任务数、模式数和Docker镜像摘要。
任何不一致都停止，避免在已改变的候选表上继续运行。

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def as_bool(series):
    # Robustly decode booleans saved by pandas/Excel/CSV.
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.isin({'true', '1', 'yes', 'y', 't'})

plan = read_json(PATHS['plan_manifest'])
plan_qc = pd.read_csv(PATHS['plan_qc'])
jobs = pd.read_csv(PATHS['generation_jobs'])
compounds = pd.read_csv(PATHS['compound_manifest'])
permissions = pd.read_csv(PATHS['fold_permissions'])
references = pd.read_csv(PATHS['reference_pairs'])

assert plan['status'] == 'READY_FOR_NOTEBOOK_15'
assert plan.get('external_test_unlocked') is False
assert as_bool(plan_qc['passed']).all()
expected_jobs_hash = plan['output_hashes']['cfm_generation_jobs.csv']
assert sha256_file(PATHS['generation_jobs']) == expected_jobs_hash, '14号任务表哈希变化。'
assert len(jobs) == int(plan['counts']['generation_command_jobs'])
assert jobs['inchikey_2d'].nunique() == int(plan['counts']['included_compounds'])
assert set(jobs['mode']) == {'positive', 'negative'}
assert not jobs.duplicated(['inchikey_2d', 'mode']).any()
assert len(references) == int(plan['counts']['expected_simulated_spectra'])
strict_reference_mask = as_bool(references['strict_public_reference_available'])
assert int(strict_reference_mask.sum()) == int(plan['counts']['strict_public_reference_pairs'])

FROZEN_IMAGE = plan['cfm']['docker_image']
FROZEN_IMAGE_DIGEST = plan['cfm']['docker_image_digest']
EXPECTED_COMMAND_JOBS = len(jobs)
EXPECTED_SPECTRA = EXPECTED_COMMAND_JOBS * 3

print('14号计划状态:', plan['status'])
print('CFM compounds:', jobs['inchikey_2d'].nunique())
print('Command jobs:', EXPECTED_COMMAND_JOBS)
print('Expected spectra:', EXPECTED_SPECTRA)
print('Frozen real references:', int(strict_reference_mask.sum()))
print('Frozen image:', FROZEN_IMAGE)
print('Frozen digest:', FROZEN_IMAGE_DIGEST)

## 2. Docker预检和CFM解析器

这里不自动下载或更新镜像。14号冻结的镜像必须已存在且digest完全一致。
`cfm-predict`每个命令输出energy0/1/2，分别映射10/20/40 eV。

In [ ]:
def run_capture(command, timeout=120):
    started = time.perf_counter()
    try:
        result = subprocess.run(
            command, capture_output=True, text=True, encoding='utf-8', errors='replace',
            timeout=timeout, shell=False,
        )
        return {
            'ok': result.returncode == 0, 'returncode': result.returncode,
            'stdout': result.stdout.strip(), 'stderr': result.stderr.strip(),
            'seconds': time.perf_counter() - started,
        }
    except Exception as exc:
        return {
            'ok': False, 'returncode': None, 'stdout': '', 'stderr': repr(exc),
            'seconds': time.perf_counter() - started,
        }

docker_executable = shutil.which('docker')
docker_client = run_capture([docker_executable, '--version']) if docker_executable else {'ok': False, 'stderr': 'docker not found', 'stdout': ''}
docker_server = run_capture([docker_executable, 'info', '--format', '{{json .ServerVersion}}']) if docker_executable else docker_client
image_result = run_capture([
    docker_executable, 'image', 'inspect', '--format', '{{json .RepoDigests}}|{{.Id}}', FROZEN_IMAGE,
]) if docker_executable and docker_server['ok'] else {'ok': False, 'stderr': 'Docker server not ready', 'stdout': ''}
actual_digest = image_result['stdout'] if image_result['ok'] else 'unavailable'
ENVIRONMENT_READY = bool(
    docker_executable and docker_client['ok'] and docker_server['ok'] and image_result['ok']
    and actual_digest == FROZEN_IMAGE_DIGEST
)
environment_qc = pd.DataFrame([
    {'check': 'docker executable', 'passed': bool(docker_executable), 'detail': str(docker_executable)},
    {'check': 'docker client', 'passed': docker_client['ok'], 'detail': (docker_client.get('stdout') or docker_client.get('stderr'))[-500:]},
    {'check': 'docker server', 'passed': docker_server['ok'], 'detail': (docker_server.get('stdout') or docker_server.get('stderr'))[-500:]},
    {'check': 'frozen image present', 'passed': image_result['ok'], 'detail': FROZEN_IMAGE},
    {'check': 'image digest exact match', 'passed': actual_digest == FROZEN_IMAGE_DIGEST, 'detail': actual_digest},
])
environment_qc.to_csv(OUTPUT_DIR / 'cfm_batch_environment_qc.csv', index=False, encoding='utf-8-sig')
display(environment_qc)
if RUN_BATCH:
    assert ENVIRONMENT_READY, 'Docker或冻结CFM镜像未通过预检。请启动Docker Desktop，不要更换镜像版本。'

ENERGY_HEADER = re.compile(r'^energy\s*([012])\s*$', flags=re.I)
PEAK_LINE = re.compile(
    r'^\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)'
    r'\s+([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)'
)

def parse_cfm_output(path):
    spectra = {0: [], 1: [], 2: []}
    current = None
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return spectra
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        header = ENERGY_HEADER.match(line.strip())
        if header:
            current = int(header.group(1))
            continue
        peak = PEAK_LINE.match(line)
        if current is not None and peak:
            mz, intensity = float(peak.group(1)), float(peak.group(2))
            if np.isfinite(mz) and np.isfinite(intensity) and mz > 0 and intensity > 0:
                spectra[current].append((mz, intensity))
    return spectra

def output_is_complete(path):
    parsed = parse_cfm_output(path)
    return all(len(parsed[index]) > 0 for index in COLLISION_ENERGY_MAP)

## 3. 批量生成（断点续跑）

每个任务启动一个固定CFM Docker容器。虽然单任务容器启动有额外开销，但该方式继承13号已验证的命令，
可以精确记录每个结构/模式的失败和耗时。状态表每10个任务安全替换写入。

In [ ]:
STATUS_PATH = OUTPUT_DIR / 'cfm_batch_execution_status.csv'
previous_status = pd.read_csv(STATUS_PATH) if STATUS_PATH.exists() else pd.DataFrame()
if len(previous_status):
    previous_status = previous_status.loc[previous_status['job_id'].isin(set(jobs['job_id']))].copy()
status_by_job = previous_status.set_index('job_id').to_dict('index') if len(previous_status) else {}

def host_output_path(row):
    return OUTPUT_DIR / row.output_relative_path

def container_output_path(row):
    return '/cfmid/public/' + Path(row.output_relative_path).as_posix()

def docker_command(row):
    mount = f'{OUTPUT_DIR.resolve()}:/cfmid/public'
    return [
        docker_executable, 'run', '--rm', '-v', mount, '-w', '/cfmid/public',
        FROZEN_IMAGE, 'cfm-predict', row.cfm_smiles,
        str(row.probability_threshold), row.model_param, row.model_config,
        '0', container_output_path(row),
    ]

def classify_failure(stderr):
    text = str(stderr).lower()
    if 'ionizationexception' in text or 'unable to proceed' in text:
        return 'ionization_failure'
    if 'timedout' in text or 'timeout' in text:
        return 'timeout'
    if 'memory' in text or 'bad_alloc' in text:
        return 'memory_failure'
    if 'segmentation' in text or 'returncode 139' in text:
        return 'process_crash'
    if 'mount' in text or 'permission' in text:
        return 'docker_mount_or_permission'
    return 'other_failure'

def execute_job(row):
    output_path = host_output_path(row)
    old = status_by_job.get(row.job_id, {})
    if output_is_complete(output_path):
        parsed = parse_cfm_output(output_path)
        return {
            'job_id': row.job_id, 'cfm_compound_id': row.cfm_compound_id,
            'inchikey_2d': row.inchikey_2d, 'mode': row.mode,
            'status': 'cached_complete', 'failure_class': '', 'returncode': 0,
            'runtime_seconds': old.get('runtime_seconds', np.nan),
            'started_at_utc': old.get('started_at_utc', ''),
            'finished_at_utc': old.get('finished_at_utc', ''),
            'output_bytes': output_path.stat().st_size,
            **{f'energy{i}_peak_n': len(parsed[i]) for i in range(3)},
            'stderr_tail': old.get('stderr_tail', ''),
            'command_json': json.dumps(docker_command(row), ensure_ascii=False),
        }
    if old and str(old.get('status')) in {'failed_or_incomplete', 'timeout_or_exception'} and not RERUN_FAILED:
        return old
    output_path.parent.mkdir(parents=True, exist_ok=True)
    started_at = utc_now()
    result = run_capture(docker_command(row), timeout=COMMAND_TIMEOUT_SECONDS)
    finished_at = utc_now()
    parsed = parse_cfm_output(output_path)
    complete = result['ok'] and all(len(parsed[index]) > 0 for index in COLLISION_ENERGY_MAP)
    stderr_tail = (result['stderr'] or result['stdout'])[-3000:]
    status = 'success' if complete else ('timeout_or_exception' if result['returncode'] is None else 'failed_or_incomplete')
    return {
        'job_id': row.job_id, 'cfm_compound_id': row.cfm_compound_id,
        'inchikey_2d': row.inchikey_2d, 'mode': row.mode,
        'status': status, 'failure_class': '' if complete else classify_failure(stderr_tail),
        'returncode': result['returncode'], 'runtime_seconds': result['seconds'],
        'started_at_utc': started_at, 'finished_at_utc': finished_at,
        'output_bytes': output_path.stat().st_size if output_path.exists() else 0,
        **{f'energy{i}_peak_n': len(parsed[i]) for i in range(3)},
        'stderr_tail': stderr_tail,
        'command_json': json.dumps(docker_command(row), ensure_ascii=False),
    }

def atomic_status_write(status_map):
    frame = pd.DataFrame([status_map[key] for key in sorted(status_map)])
    temporary = STATUS_PATH.with_suffix('.csv.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8-sig')
    os.replace(temporary, STATUS_PATH)
    return frame

if RUN_BATCH:
    assert MAX_JOBS is None, '正式批量运行必须保持MAX_JOBS=None。'
    rows_to_run = jobs
    for index, row in enumerate(tqdm(rows_to_run.itertuples(index=False), total=len(rows_to_run), desc='CFM batch')):
        status_by_job[row.job_id] = execute_job(row)
        if (index + 1) % SAVE_STATUS_EVERY == 0 or index + 1 == len(rows_to_run):
            execution_status = atomic_status_write(status_by_job)
elif STATUS_PATH.exists():
    execution_status = pd.read_csv(STATUS_PATH)
    print('RUN_BATCH=False：仅读取已有任务状态，不启动CFM。')
else:
    execution_status = pd.DataFrame(columns=['job_id', 'mode', 'status'])
    print('RUN_BATCH=False且没有已有状态。')

if len(execution_status):
    execution_summary = execution_status.groupby(['mode', 'status'], dropna=False).size().rename('n').reset_index()
    display(execution_summary)
    print('Status rows:', len(execution_status), '/', EXPECTED_COMMAND_JOBS)

## 4. 解析模拟谱并执行硬性物理QC

对每个成功命令要求三能量谱均非空、m/z与强度有限且为正、碎片不高于理论离子母体超过0.01 Da。
峰数少于3不直接删除，而标记为`sparse_spectrum`，供15号和后续Transformer做敏感性分析。

In [ ]:
PROTON_MASS = 1.007276466621

def theoretical_precursor(smiles, mode):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.nan
    neutral_mass = float(Descriptors.ExactMolWt(mol))
    return neutral_mass + PROTON_MASS if mode == 'positive' else neutral_mass - PROTON_MASS

job_precursor = {
    row.job_id: theoretical_precursor(row.cfm_smiles, row.mode)
    for row in jobs.itertuples(index=False)
}
peak_rows = []
spectrum_rows = []
raw_hash_rows = []
for row in tqdm(jobs.itertuples(index=False), total=len(jobs), desc='Parse and physical QC'):
    output_path = host_output_path(row)
    parsed = parse_cfm_output(output_path)
    if output_path.exists() and output_path.stat().st_size > 0:
        raw_hash_rows.append({
            'job_id': row.job_id,
            'output_relative_path': row.output_relative_path,
            'size_bytes': output_path.stat().st_size,
            'sha256': sha256_file(output_path),
        })
    precursor = job_precursor[row.job_id]
    for energy_index, energy_ev in COLLISION_ENERGY_MAP.items():
        peaks = sorted(parsed[energy_index])
        mz = np.asarray([item[0] for item in peaks], dtype=float)
        intensity = np.asarray([item[1] for item in peaks], dtype=float)
        finite_positive = bool(len(peaks) and np.isfinite(mz).all() and np.isfinite(intensity).all() and (mz > 0).all() and (intensity > 0).all())
        fragment_below_precursor = bool(len(peaks) and np.isfinite(precursor) and (mz <= precursor + PRECURSOR_MZ_TOLERANCE_DA).all())
        unique_mz = bool(len(peaks) and (len(mz) == 1 or (np.diff(mz) > 1e-8).all()))
        parse_success = len(peaks) > 0
        hard_qc_pass = parse_success and finite_positive and fragment_below_precursor and unique_mz
        maximum = float(intensity.max()) if len(intensity) else np.nan
        task_id = f'{row.job_id}_ce{energy_ev}'
        spectrum_rows.append({
            'spectrum_task_id': task_id, 'job_id': row.job_id,
            'cfm_compound_id': row.cfm_compound_id, 'inchikey_2d': row.inchikey_2d,
            'generation_role': row.generation_role, 'mode': row.mode, 'adduct': row.adduct,
            'energy_index': energy_index, 'collision_energy_ev': energy_ev,
            'theoretical_precursor_mz': precursor, 'peak_n': len(peaks),
            'max_fragment_mz': float(mz.max()) if len(mz) else np.nan,
            'parse_success': parse_success, 'finite_positive': finite_positive,
            'fragment_below_precursor': fragment_below_precursor, 'unique_mz': unique_mz,
            'sparse_spectrum_lt3_peaks': 0 < len(peaks) < 3,
            'hard_qc_pass': hard_qc_pass,
        })
        for peak_rank, (fragment_mz, raw_intensity) in enumerate(peaks, start=1):
            peak_rows.append({
                'spectrum_task_id': task_id, 'job_id': row.job_id,
                'cfm_compound_id': row.cfm_compound_id, 'inchikey_2d': row.inchikey_2d,
                'generation_role': row.generation_role, 'mode': row.mode, 'adduct': row.adduct,
                'collision_energy_ev': energy_ev, 'peak_rank_mz': peak_rank,
                'fragment_mz': fragment_mz, 'intensity_raw': raw_intensity,
                'relative_intensity': raw_intensity / maximum if maximum > 0 else np.nan,
                'neutral_loss': precursor - fragment_mz,
            })

spectrum_index = pd.DataFrame(spectrum_rows)
batch_peaks = pd.DataFrame(
    peak_rows,
    columns=[
        'spectrum_task_id', 'job_id', 'cfm_compound_id', 'inchikey_2d',
        'generation_role', 'mode', 'adduct', 'collision_energy_ev',
        'peak_rank_mz', 'fragment_mz', 'intensity_raw', 'relative_intensity', 'neutral_loss',
    ],
)
raw_hashes = pd.DataFrame(
    raw_hash_rows,
    columns=['job_id', 'output_relative_path', 'size_bytes', 'sha256'],
)
spectrum_index.to_csv(OUTPUT_DIR / 'cfm_batch_spectrum_index.csv', index=False, encoding='utf-8-sig')
raw_hashes.to_csv(OUTPUT_DIR / 'cfm_raw_output_hashes.csv', index=False, encoding='utf-8-sig')
if len(batch_peaks):
    batch_peaks.to_csv(OUTPUT_DIR / 'cfm_batch_peaks.csv.gz', index=False, compression='gzip')

failed_jobs = execution_status.loc[~execution_status['status'].isin(['success', 'cached_complete'])].copy() if len(execution_status) else pd.DataFrame()
failed_jobs.to_csv(OUTPUT_DIR / 'cfm_batch_failed_jobs.csv', index=False, encoding='utf-8-sig')
physical_failures = spectrum_index.loc[spectrum_index['parse_success'] & ~spectrum_index['hard_qc_pass']].copy()
physical_failures.to_csv(OUTPUT_DIR / 'cfm_batch_physical_qc_failures.csv', index=False, encoding='utf-8-sig')

print('Parsed spectra:', int(spectrum_index['parse_success'].sum()), '/', EXPECTED_SPECTRA)
print('Parsed peaks:', len(batch_peaks))
print('Hard physical-QC failures:', len(physical_failures))
print('Sparse spectra (<3 peaks):', int(spectrum_index['sparse_spectrum_lt3_peaks'].sum()))
if len(failed_jobs):
    display(failed_jobs.groupby(['mode', 'failure_class'], dropna=False).size().rename('n').reset_index())

## 5. 载入冻结的公共实测对照

只读取14号预先选定、且对应成功模拟谱的公共Development记录。逐条校验身份、来源和峰数；任何ChemVista来源立即停止。

In [ ]:
successful_task_ids = set(spectrum_index.loc[spectrum_index['hard_qc_pass'], 'spectrum_task_id'].astype(str))
strict_references = references.loc[strict_reference_mask].copy()
strict_references['spectrum_task_id'] = strict_references['job_id'].astype(str) + '_ce' + strict_references['simulated_ce_ev'].astype(int).astype(str)
audit_pairs = strict_references.loc[strict_references['spectrum_task_id'].isin(successful_task_ids)].copy()

public_manifest = pd.read_csv(PATHS['public_manifest'])
public_lookup = public_manifest.set_index('spectrum_id', drop=False)
required_real_ids = sorted(set(audit_pairs['measured_spectrum_id'].astype(str)))
missing_real_ids = sorted(set(required_real_ids) - set(public_lookup.index.astype(str)))
assert not missing_real_ids, f'冻结实测谱不在公共清单：{missing_real_ids[:5]}'

MAX_MZ = 2000.0
def extract_observed_peaks(record):
    tensor = np.asarray(record['tensor'], dtype=np.float64)
    padding = np.asarray(record['padding_mask'], dtype=bool)
    if tensor.ndim != 2 or tensor.shape[1] < 4 or padding.shape != (len(tensor),):
        raise ValueError('Unexpected trusted public spectrum schema')
    valid = ~padding
    valid[0] = False
    mz = tensor[valid, 0] * MAX_MZ
    intensity = tensor[valid, 3]
    keep = np.isfinite(mz) & np.isfinite(intensity) & (mz > 0) & (intensity > 0)
    peaks = np.column_stack([mz[keep], intensity[keep]])
    return peaks[np.argsort(peaks[:, 0])] if len(peaks) else peaks

measured_cache = {}
measured_meta = {}
rows_to_load = public_lookup.loc[required_real_ids].sort_values('stream_offset') if required_real_ids else pd.DataFrame()
with PATHS['public_stream'].open('rb') as stream:
    for row in tqdm(rows_to_load.itertuples(index=False), total=len(rows_to_load), desc='Load public measured references'):
        stream.seek(int(row.stream_offset))
        actual_key, record = pickle.load(stream)
        assert str(actual_key) == str(row.inchikey_2d)
        assert str(record.get('source_library')) != 'ChemVista'
        peaks = extract_observed_peaks(record)
        assert len(peaks) == int(row.cleaned_peak_n)
        measured_cache[str(row.spectrum_id)] = peaks
        measured_meta[str(row.spectrum_id)] = {
            'inchikey_2d': str(row.inchikey_2d),
            'precursor_mz': float(row.precursor_mz),
            'collision_energy_value': float(row.collision_energy_value),
            'mode': str(row.ion_mode).lower(),
            'adduct': str(row.adduct),
            'source_library': str(row.source_library),
        }

sim_peak_cache = {
    key: group[['fragment_mz', 'relative_intensity']].to_numpy(float)
    for key, group in batch_peaks.groupby('spectrum_task_id', sort=False)
}
assert set(audit_pairs['measured_spectrum_id'].astype(str)) <= set(measured_cache)
assert set(audit_pairs['spectrum_task_id'].astype(str)) <= set(sim_peak_cache)
print('Strict frozen pairs:', len(strict_references))
print('Pairs with successful hard-QC CFM spectra:', len(audit_pairs))
print('Unique public measured spectra loaded:', len(measured_cache))

## 6. 谱图配对指标

- **Modified cosine**：平方根强度、0.02 Da容差，同时允许直接峰匹配和理论/实测母离子差对应的平移匹配。
- **Entropy similarity**：在同一贪心峰对齐上计算归一化Jensen–Shannon谱熵相似度。
- **Measured peak recall**：实测峰中被CFM覆盖的比例。
- **Simulated peak precision**：CFM峰中能在实测谱找到的比例。
- **Intensity correlation**：至少3个匹配峰时计算Pearson和Spearman。
- **Binned cosine**：1 Da固定网格的谱图向量余弦，作为非深度嵌入距离基线。

In [ ]:
def normalize_intensity(peaks, sqrt_transform=False):
    peaks = np.asarray(peaks, dtype=float)
    if len(peaks) == 0:
        return peaks
    values = np.clip(peaks[:, 1], 0, None)
    if sqrt_transform:
        values = np.sqrt(values)
    total = np.linalg.norm(values)
    if total <= 0:
        return np.column_stack([peaks[:, 0], np.zeros(len(peaks))])
    return np.column_stack([peaks[:, 0], values / total])

def greedy_modified_alignment(sim_peaks, real_peaks, precursor_shift, tolerance=MZ_TOLERANCE_DA):
    sim_norm = normalize_intensity(sim_peaks, sqrt_transform=True)
    real_norm = normalize_intensity(real_peaks, sqrt_transform=True)
    candidates = []
    for i, (sim_mz, sim_i) in enumerate(sim_norm):
        for shifted, match_type in ((sim_mz, 'direct'), (sim_mz + precursor_shift, 'precursor_shift')):
            left = np.searchsorted(real_norm[:, 0], shifted - tolerance, side='left')
            right = np.searchsorted(real_norm[:, 0], shifted + tolerance, side='right')
            for j in range(left, right):
                candidates.append((float(sim_i * real_norm[j, 1]), -abs(real_norm[j, 0] - shifted), i, j, match_type))
    used_sim, used_real, matches = set(), set(), []
    for contribution, neg_error, i, j, match_type in sorted(candidates, reverse=True):
        if i not in used_sim and j not in used_real:
            used_sim.add(i); used_real.add(j)
            matches.append((i, j, contribution, -neg_error, match_type))
    return matches

def entropy_similarity_from_alignment(sim_peaks, real_peaks, matches):
    sim_intensity = np.clip(np.asarray(sim_peaks)[:, 1], 0, None)
    real_intensity = np.clip(np.asarray(real_peaks)[:, 1], 0, None)
    sim_intensity = sim_intensity / sim_intensity.sum()
    real_intensity = real_intensity / real_intensity.sum()
    matched_sim = {item[0] for item in matches}
    matched_real = {item[1] for item in matches}
    p, q = [], []
    for i, j, *_ in matches:
        p.append(sim_intensity[i]); q.append(real_intensity[j])
    for i in range(len(sim_peaks)):
        if i not in matched_sim:
            p.append(sim_intensity[i]); q.append(0.0)
    for j in range(len(real_peaks)):
        if j not in matched_real:
            p.append(0.0); q.append(real_intensity[j])
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float); middle = 0.5 * (p + q)
    kl_p = np.sum(np.where(p > 0, p * np.log(p / middle), 0.0))
    kl_q = np.sum(np.where(q > 0, q * np.log(q / middle), 0.0))
    jsd = 0.5 * (kl_p + kl_q)
    return float(np.clip(1.0 - jsd / np.log(2.0), 0.0, 1.0))

def binned_vector(peaks, bin_width=DOMAIN_BIN_WIDTH_DA, max_mz=DOMAIN_MAX_MZ):
    n_bins = int(math.ceil(max_mz / bin_width))
    vector = np.zeros(n_bins, dtype=np.float32)
    peaks = np.asarray(peaks, dtype=float)
    if len(peaks):
        indices = np.floor(peaks[:, 0] / bin_width).astype(int)
        valid = (indices >= 0) & (indices < n_bins) & np.isfinite(peaks[:, 1]) & (peaks[:, 1] > 0)
        np.add.at(vector, indices[valid], np.sqrt(peaks[valid, 1]).astype(np.float32))
        norm = np.linalg.norm(vector)
        if norm > 0:
            vector /= norm
    return vector

def paired_metrics(sim_peaks, real_peaks, sim_precursor, real_precursor):
    precursor_shift = real_precursor - sim_precursor
    matches = greedy_modified_alignment(sim_peaks, real_peaks, precursor_shift)
    modified_cosine = float(sum(item[2] for item in matches))
    sim_indices = [item[0] for item in matches]
    real_indices = [item[1] for item in matches]
    if len(matches) >= 3:
        sim_values = np.asarray(sim_peaks)[sim_indices, 1]
        real_values = np.asarray(real_peaks)[real_indices, 1]
        pearson = float(pearsonr(sim_values, real_values).statistic)
        spearman = float(spearmanr(sim_values, real_values).statistic)
    else:
        pearson = np.nan; spearman = np.nan
    sim_vector = binned_vector(sim_peaks)
    real_vector = binned_vector(real_peaks)
    return {
        'modified_cosine': modified_cosine,
        'spectral_entropy_similarity': entropy_similarity_from_alignment(sim_peaks, real_peaks, matches),
        'matched_peak_n': len(matches),
        'measured_peak_recall': len(set(real_indices)) / len(real_peaks),
        'simulated_peak_precision': len(set(sim_indices)) / len(sim_peaks),
        'matched_intensity_pearson': pearson,
        'matched_intensity_spearman': spearman,
        'binned_cosine': float(np.dot(sim_vector, real_vector)),
        'binned_embedding_distance': float(1.0 - np.dot(sim_vector, real_vector)),
        'direct_match_n': sum(item[4] == 'direct' for item in matches),
        'shifted_match_n': sum(item[4] == 'precursor_shift' for item in matches),
        'median_match_mass_error_da': float(np.median([item[3] for item in matches])) if matches else np.nan,
    }

In [ ]:
pair_rows = []
spectrum_meta = spectrum_index.set_index('spectrum_task_id')
for pair in tqdm(audit_pairs.itertuples(index=False), total=len(audit_pairs), desc='Sim-to-real metrics'):
    sim_peaks = sim_peak_cache[str(pair.spectrum_task_id)]
    real_peaks = measured_cache[str(pair.measured_spectrum_id)]
    sim_meta = spectrum_meta.loc[str(pair.spectrum_task_id)]
    real_meta = measured_meta[str(pair.measured_spectrum_id)]
    assert str(pair.inchikey_2d) == str(real_meta['inchikey_2d'])
    assert str(pair.mode) == str(real_meta['mode'])
    assert str(pair.adduct) == str(real_meta['adduct'])
    metrics = paired_metrics(
        sim_peaks, real_peaks,
        float(sim_meta['theoretical_precursor_mz']), float(real_meta['precursor_mz']),
    )
    pair_rows.append({
        'spectrum_task_id': pair.spectrum_task_id,
        'job_id': pair.job_id, 'cfm_compound_id': pair.cfm_compound_id,
        'inchikey_2d': pair.inchikey_2d, 'mode': pair.mode, 'adduct': pair.adduct,
        'simulated_ce_ev': int(pair.simulated_ce_ev),
        'measured_spectrum_id': pair.measured_spectrum_id,
        'measured_ce_ev': float(pair.measured_ce_ev), 'ce_delta_ev': float(pair.ce_delta_ev),
        'simulated_precursor_mz': float(sim_meta['theoretical_precursor_mz']),
        'measured_precursor_mz': float(real_meta['precursor_mz']),
        'precursor_delta_ppm': 1e6 * (float(real_meta['precursor_mz']) - float(sim_meta['theoretical_precursor_mz'])) / float(sim_meta['theoretical_precursor_mz']),
        'simulated_peak_n': len(sim_peaks), 'measured_peak_n': len(real_peaks),
        'log_peak_count_ratio': math.log((len(sim_peaks) + 1) / (len(real_peaks) + 1)),
        'simulated_max_mz_ratio': float(np.max(sim_peaks[:, 0]) / sim_meta['theoretical_precursor_mz']),
        'measured_max_mz_ratio': float(np.max(real_peaks[:, 0]) / real_meta['precursor_mz']),
        **metrics,
    })

sim_to_real = pd.DataFrame(pair_rows)
sim_to_real.to_csv(OUTPUT_DIR / 'cfm_sim_to_real_metrics.csv', index=False, encoding='utf-8-sig')
metric_columns = [
    'modified_cosine', 'spectral_entropy_similarity', 'measured_peak_recall',
    'simulated_peak_precision', 'matched_intensity_spearman', 'binned_cosine',
]
if len(sim_to_real):
    summary_rows = []
    for (mode, energy), group in sim_to_real.groupby(['mode', 'simulated_ce_ev']):
        for metric in metric_columns:
            values = pd.to_numeric(group[metric], errors='coerce').dropna()
            summary_rows.append({
                'mode': mode, 'simulated_ce_ev': energy, 'metric': metric,
                'n': len(values), 'mean': values.mean(), 'median': values.median(),
                'q25': values.quantile(0.25), 'q75': values.quantile(0.75),
            })
    sim_to_real_summary = pd.DataFrame(summary_rows)
else:
    sim_to_real_summary = pd.DataFrame(columns=['mode', 'simulated_ce_ev', 'metric', 'n', 'mean', 'median', 'q25', 'q75'])
sim_to_real_summary.to_csv(OUTPUT_DIR / 'cfm_sim_to_real_summary.csv', index=False)
print('Completed sim-to-real pairs:', len(sim_to_real))
display(sim_to_real_summary.loc[sim_to_real_summary['metric'].isin(['modified_cosine', 'spectral_entropy_similarity'])])

## 7. Real-vs-synthetic域分类器

使用1 Da平方根强度谱图向量和离子模式，按二维InChIKey做5折GroupKFold。分类器从未看到测试折化合物；
因此高AUC表示跨结构仍存在稳定的合成域特征，而不是记住同一化合物。

这是固定的诊断模型（L2 LogisticRegression，C=1），不做超参数搜索。

In [ ]:
domain_samples = []
domain_vectors = []
for task_id in sorted(set(audit_pairs['spectrum_task_id'].astype(str))):
    meta = spectrum_meta.loc[task_id]
    domain_samples.append({
        'sample_id': 'SIM:' + task_id, 'inchikey_2d': str(meta['inchikey_2d']),
        'domain': 1, 'mode': str(meta['mode']), 'source_id': task_id,
    })
    domain_vectors.append(binned_vector(sim_peak_cache[task_id]))
for spectrum_id in sorted(measured_cache):
    meta = measured_meta[spectrum_id]
    domain_samples.append({
        'sample_id': 'REAL:' + spectrum_id, 'inchikey_2d': meta['inchikey_2d'],
        'domain': 0, 'mode': meta['mode'], 'source_id': spectrum_id,
    })
    domain_vectors.append(binned_vector(measured_cache[spectrum_id]))

domain_samples = pd.DataFrame(domain_samples)
if len(domain_samples):
    X_spectra = sparse.csr_matrix(np.vstack(domain_vectors), dtype=np.float32)
    mode_feature = sparse.csr_matrix(domain_samples['mode'].eq('positive').astype(np.float32).to_numpy()[:, None])
    X_domain = sparse.hstack([X_spectra, mode_feature], format='csr')
    y_domain = domain_samples['domain'].to_numpy(int)
    groups = domain_samples['inchikey_2d'].astype(str).to_numpy()
    splitter = GroupKFold(n_splits=DOMAIN_CV_SPLITS)
    domain_probability = np.full(len(domain_samples), np.nan, dtype=float)
    domain_fold = np.full(len(domain_samples), -1, dtype=int)
    for fold, (train_index, test_index) in enumerate(splitter.split(X_domain, y_domain, groups)):
        model = LogisticRegression(
            C=1.0, penalty='l2', solver='liblinear', class_weight='balanced',
            max_iter=2000, random_state=SEED + fold,
        )
        model.fit(X_domain[train_index], y_domain[train_index])
        domain_probability[test_index] = model.predict_proba(X_domain[test_index])[:, 1]
        domain_fold[test_index] = fold
    domain_samples['outer_fold'] = domain_fold
    domain_samples['synthetic_probability'] = domain_probability
    domain_samples['predicted_domain'] = (domain_probability >= 0.5).astype(int)
    domain_metrics = pd.DataFrame([{
        'model': '1Da_binned_spectrum_L2_logistic_group5fold',
        'n_samples': len(domain_samples), 'n_compounds': domain_samples['inchikey_2d'].nunique(),
        'n_synthetic': int(y_domain.sum()), 'n_real': int((1 - y_domain).sum()),
        'roc_auc': roc_auc_score(y_domain, domain_probability),
        'pr_auc_synthetic': average_precision_score(y_domain, domain_probability),
        'balanced_accuracy': balanced_accuracy_score(y_domain, domain_probability >= 0.5),
    }])
else:
    domain_samples['outer_fold'] = []
    domain_samples['synthetic_probability'] = []
    domain_samples['predicted_domain'] = []
    domain_metrics = pd.DataFrame(columns=['model', 'n_samples', 'n_compounds', 'n_synthetic', 'n_real', 'roc_auc', 'pr_auc_synthetic', 'balanced_accuracy'])

domain_samples.to_csv(OUTPUT_DIR / 'cfm_domain_classifier_oof.csv.gz', index=False, compression='gzip')
domain_metrics.to_csv(OUTPUT_DIR / 'cfm_domain_classifier_metrics.csv', index=False)
display(domain_metrics)

## 8. 预先约定的CFM使用决策

总体决策规则：

- 域分类ROC-AUC ≥0.90或modified cosine中位数<0.40：只用于预训练/低权重一致性，不能与真实谱同权监督。
- 域AUC 0.75–0.90或cosine中位数0.40–0.65：可测试低权重增强，必须保留synthetic标记。
- 域AUC<0.75且cosine中位数≥0.65：可作为同权增强候选，但仍需16–18号消融验证。

分能量建议仅依据Development公共实测谱，不依据ChemVista。

In [ ]:
if len(sim_to_real) and len(domain_metrics):
    overall_cosine_median = float(sim_to_real['modified_cosine'].median())
    domain_auc = float(domain_metrics['roc_auc'].iloc[0])
    if domain_auc >= 0.90 or overall_cosine_median < 0.40:
        overall_recommendation = 'PRETRAINING_OR_LOW_WEIGHT_CONSISTENCY_ONLY'
    elif domain_auc >= 0.75 or overall_cosine_median < 0.65:
        overall_recommendation = 'LOW_WEIGHT_AUGMENTATION_WITH_DOMAIN_LABEL'
    else:
        overall_recommendation = 'EQUAL_WEIGHT_AUGMENTATION_CANDIDATE_REQUIRES_ABLATION'
    energy_rows = []
    for (mode, energy), group in sim_to_real.groupby(['mode', 'simulated_ce_ev']):
        median_cosine = float(group['modified_cosine'].median())
        if median_cosine >= 0.65:
            recommendation = 'augmentation_candidate'
        elif median_cosine >= 0.40:
            recommendation = 'low_weight_or_pretraining'
        else:
            recommendation = 'pretraining_only'
        energy_rows.append({
            'mode': mode, 'collision_energy_ev': energy,
            'n_pairs': len(group), 'median_modified_cosine': median_cosine,
            'recommendation': recommendation,
        })
    energy_decisions = pd.DataFrame(energy_rows)
    domain_decision = pd.DataFrame([{
        'overall_modified_cosine_median': overall_cosine_median,
        'domain_classifier_roc_auc': domain_auc,
        'overall_recommendation': overall_recommendation,
    }])
else:
    overall_recommendation = 'BATCH_OR_AUDIT_INCOMPLETE'
    energy_decisions = pd.DataFrame(columns=['mode', 'collision_energy_ev', 'n_pairs', 'median_modified_cosine', 'recommendation'])
    domain_decision = pd.DataFrame([{
        'overall_modified_cosine_median': np.nan,
        'domain_classifier_roc_auc': np.nan,
        'overall_recommendation': overall_recommendation,
    }])
energy_decisions.to_csv(OUTPUT_DIR / 'cfm_energy_specific_decisions.csv', index=False)
domain_decision.to_csv(OUTPUT_DIR / 'cfm_domain_decision.csv', index=False)
display(domain_decision)
display(energy_decisions)

## 9. 论文图：生成稳定性、sim-to-real差距与域可分性

In [ ]:
if len(execution_status):
    success_flag = execution_status['status'].isin(['success', 'cached_complete'])
    success_plot = execution_status.assign(success=np.where(success_flag, 'success', 'failure'))
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    sns.countplot(data=success_plot, x='mode', hue='success', ax=axes[0])
    axes[0].set_title('CFM generation outcomes')
    if len(sim_to_real):
        sns.boxplot(data=sim_to_real, x='simulated_ce_ev', y='modified_cosine', hue='mode', ax=axes[1])
        axes[1].set_ylim(0, 1)
        axes[1].set_title('Modified cosine by energy')
        sns.boxplot(data=sim_to_real, x='simulated_ce_ev', y='spectral_entropy_similarity', hue='mode', ax=axes[2])
        axes[2].set_ylim(0, 1)
        axes[2].set_title('Spectral-entropy similarity')
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / '15_generation_and_sim_to_real.png', dpi=240, bbox_inches='tight')
    plt.show()

if len(domain_samples):
    fpr, tpr, _ = roc_curve(domain_samples['domain'], domain_samples['synthetic_probability'])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].plot(fpr, tpr, linewidth=2, label=f"AUC={domain_metrics['roc_auc'].iloc[0]:.3f}")
    axes[0].plot([0, 1], [0, 1], '--', color='grey')
    axes[0].set(xlabel='False-positive rate', ylabel='True-positive rate', title='Real-vs-synthetic domain classifier')
    axes[0].legend()
    sns.histplot(data=domain_samples, x='synthetic_probability', hue='domain', bins=30,
                 stat='density', common_norm=False, element='step', ax=axes[1])
    axes[1].set_title('OOF domain probabilities (0=real, 1=synthetic)')
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / '15_domain_classifier.png', dpi=240, bbox_inches='tight')
    plt.show()

## 10. 最终QC、状态和可复现运行清单

允许少量可解释的CFM生成失败，但必须满足总体≥95%、每种模式≥90%，且所有成功谱通过硬性物理QC。
如果任务未全部尝试，状态为`BATCH_INCOMPLETE_RESUME`，应继续断点运行，而不是进入16号。

In [ ]:
status_job_ids = set(execution_status['job_id'].astype(str)) if len(execution_status) else set()
all_jobs_accounted = status_job_ids == set(jobs['job_id'].astype(str))
success_flag = execution_status['status'].isin(['success', 'cached_complete']) if len(execution_status) else pd.Series(dtype=bool)
overall_success_rate = float(success_flag.mean()) if len(success_flag) else np.nan
mode_rates = execution_status.assign(success=success_flag).groupby('mode')['success'].mean() if len(execution_status) else pd.Series(dtype=float)
parsed_spectra_n = int(spectrum_index['parse_success'].sum())
successful_commands_n = int(success_flag.sum()) if len(success_flag) else 0
expected_parsed_from_success = successful_commands_n * 3
hard_failure_n = len(physical_failures)
generated_pair_coverage = len(audit_pairs) / max(len(strict_references), 1)

qc = pd.DataFrame([
    {'check': 'frozen_plan_ready', 'passed': plan['status'] == 'READY_FOR_NOTEBOOK_15', 'observed': plan['status'], 'expected': 'READY_FOR_NOTEBOOK_15'},
    {'check': 'external_test_remains_locked', 'passed': plan.get('external_test_unlocked') is False, 'observed': plan.get('external_test_unlocked'), 'expected': False},
    {'check': 'docker_image_digest_exact_match', 'passed': actual_digest == FROZEN_IMAGE_DIGEST, 'observed': actual_digest, 'expected': FROZEN_IMAGE_DIGEST},
    {'check': 'all_generation_jobs_accounted', 'passed': all_jobs_accounted, 'observed': len(status_job_ids), 'expected': len(jobs)},
    {'check': 'parsed_spectra_equal_successful_commands_times_three', 'passed': parsed_spectra_n == expected_parsed_from_success, 'observed': parsed_spectra_n, 'expected': expected_parsed_from_success},
    {'check': 'hard_physical_qc_failures_zero', 'passed': hard_failure_n == 0, 'observed': hard_failure_n, 'expected': 0},
    {'check': 'overall_generation_success_ge_95pct', 'passed': bool(np.isfinite(overall_success_rate) and overall_success_rate >= MIN_OVERALL_GENERATION_SUCCESS), 'observed': overall_success_rate, 'expected': f'>={MIN_OVERALL_GENERATION_SUCCESS}'},
    {'check': 'positive_generation_success_ge_90pct', 'passed': bool(mode_rates.get('positive', 0) >= MIN_MODE_GENERATION_SUCCESS), 'observed': mode_rates.get('positive', np.nan), 'expected': f'>={MIN_MODE_GENERATION_SUCCESS}'},
    {'check': 'negative_generation_success_ge_90pct', 'passed': bool(mode_rates.get('negative', 0) >= MIN_MODE_GENERATION_SUCCESS), 'observed': mode_rates.get('negative', np.nan), 'expected': f'>={MIN_MODE_GENERATION_SUCCESS}'},
    {'check': 'sim_to_real_pairs_present', 'passed': len(sim_to_real) > 0, 'observed': len(sim_to_real), 'expected': '>0'},
    {'check': 'domain_classifier_completed', 'passed': len(domain_metrics) == 1 and np.isfinite(pd.to_numeric(domain_metrics['roc_auc'], errors='coerce')).all(), 'observed': len(domain_metrics), 'expected': 1},
    {'check': 'every_raw_hash_maps_to_frozen_job', 'passed': set(raw_hashes['job_id'].astype(str)) <= set(jobs['job_id'].astype(str)), 'observed': len(raw_hashes), 'expected': '<= frozen jobs'},
])
qc.to_csv(OUTPUT_DIR / 'cfm_batch_audit_qc.csv', index=False, encoding='utf-8-sig')

generation_gate_names = [
    'frozen_plan_ready', 'external_test_remains_locked', 'docker_image_digest_exact_match',
    'all_generation_jobs_accounted', 'parsed_spectra_equal_successful_commands_times_three',
    'hard_physical_qc_failures_zero', 'overall_generation_success_ge_95pct',
    'positive_generation_success_ge_90pct', 'negative_generation_success_ge_90pct',
    'sim_to_real_pairs_present', 'domain_classifier_completed', 'every_raw_hash_maps_to_frozen_job',
]
if not all_jobs_accounted:
    BATCH_STATUS = 'BATCH_INCOMPLETE_RESUME'
elif qc.loc[qc['check'].isin(generation_gate_names), 'passed'].all():
    BATCH_STATUS = 'READY_FOR_NOTEBOOK_16'
else:
    BATCH_STATUS = 'BLOCKED_REVIEW_QC'
display(qc)
print('BATCH_STATUS:', BATCH_STATUS)
print('CFM_USAGE_RECOMMENDATION:', overall_recommendation)

In [ ]:
formal_outputs = [
    OUTPUT_DIR / 'cfm_batch_environment_qc.csv',
    OUTPUT_DIR / 'cfm_batch_execution_status.csv',
    OUTPUT_DIR / 'cfm_batch_spectrum_index.csv',
    OUTPUT_DIR / 'cfm_raw_output_hashes.csv',
    OUTPUT_DIR / 'cfm_batch_failed_jobs.csv',
    OUTPUT_DIR / 'cfm_batch_physical_qc_failures.csv',
    OUTPUT_DIR / 'cfm_sim_to_real_metrics.csv',
    OUTPUT_DIR / 'cfm_sim_to_real_summary.csv',
    OUTPUT_DIR / 'cfm_domain_classifier_oof.csv.gz',
    OUTPUT_DIR / 'cfm_domain_classifier_metrics.csv',
    OUTPUT_DIR / 'cfm_energy_specific_decisions.csv',
    OUTPUT_DIR / 'cfm_domain_decision.csv',
    OUTPUT_DIR / 'cfm_batch_audit_qc.csv',
]
if (OUTPUT_DIR / 'cfm_batch_peaks.csv.gz').exists():
    formal_outputs.append(OUTPUT_DIR / 'cfm_batch_peaks.csv.gz')
formal_outputs.extend(sorted(FIGURE_DIR.glob('*.png')))
existing_outputs = [path for path in formal_outputs if path.exists()]

manifest = {
    'notebook': '15_CFM批量谱图整合与Sim_to_Real审计.ipynb',
    'generated_at_utc': utc_now(),
    'status': BATCH_STATUS,
    'cfm_usage_recommendation': overall_recommendation,
    'scientific_scope': 'Frozen CFM generation and public-Development sim-to-real audit; ChemVista remains locked',
    'external_test_unlocked': False,
    'frozen_plan': {
        'manifest_path': str(PATHS['plan_manifest']),
        'manifest_sha256': sha256_file(PATHS['plan_manifest']),
        'generation_jobs_sha256': sha256_file(PATHS['generation_jobs']),
        'docker_image': FROZEN_IMAGE, 'docker_image_digest': FROZEN_IMAGE_DIGEST,
    },
    'counts': {
        'command_jobs_expected': EXPECTED_COMMAND_JOBS,
        'command_jobs_accounted': len(status_job_ids),
        'command_jobs_successful': successful_commands_n,
        'command_jobs_failed': int(len(execution_status) - successful_commands_n) if len(execution_status) else 0,
        'spectra_expected': EXPECTED_SPECTRA,
        'spectra_parsed': parsed_spectra_n,
        'fragment_peaks_parsed': len(batch_peaks),
        'sparse_spectra_lt3_peaks': int(spectrum_index['sparse_spectrum_lt3_peaks'].sum()),
        'hard_physical_qc_failures': hard_failure_n,
        'frozen_strict_real_pairs': len(strict_references),
        'completed_sim_to_real_pairs': len(sim_to_real),
        'sim_to_real_pair_coverage': generated_pair_coverage,
    },
    'generation_rates': {
        'overall': overall_success_rate,
        'positive': mode_rates.get('positive', np.nan),
        'negative': mode_rates.get('negative', np.nan),
    },
    'sim_to_real': {
        'mz_tolerance_da': MZ_TOLERANCE_DA,
        'max_real_ce_delta_ev': MAX_REAL_CE_DELTA_EV,
        'metrics': ['modified_cosine', 'spectral_entropy_similarity', 'measured_peak_recall',
                    'simulated_peak_precision', 'matched_intensity_correlation',
                    'binned_cosine', 'binned_embedding_distance'],
        'overall_modified_cosine_median': float(sim_to_real['modified_cosine'].median()) if len(sim_to_real) else np.nan,
    },
    'domain_classifier': domain_metrics.iloc[0].to_dict() if len(domain_metrics) else {},
    'software': {
        'python': sys.version, 'platform': platform.platform(), 'rdkit': rdBase.rdkitVersion,
        'docker_client': docker_client.get('stdout', ''), 'docker_server': docker_server.get('stdout', ''),
    },
    'input_hashes': {name: sha256_file(path) for name, path in PATHS.items() if name != 'public_stream'},
    'public_stream_provenance': {
        'path': str(PATHS['public_stream']), 'size_bytes': PATHS['public_stream'].stat().st_size,
        'records_verified_against_frozen_manifest': True,
    },
    'output_hashes': {str(path.relative_to(OUTPUT_DIR)): sha256_file(path) for path in existing_outputs},
}
manifest_path = OUTPUT_DIR / 'cfm_batch_run_manifest.json'
temporary = manifest_path.with_suffix('.json.tmp')
temporary.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
os.replace(temporary, manifest_path)
print('Manifest:', manifest_path)
if BATCH_STATUS == 'READY_FOR_NOTEBOOK_16':
    print('15号完成，可以进入16号Transformer数据冻结。')
elif BATCH_STATUS == 'BATCH_INCOMPLETE_RESUME':
    print('任务尚未全部记录；保持配置不变，从头重跑即可断点续跑。')
else:
    print('存在未通过QC；先检查失败表和物理QC，不得进入16号。')

## 下一步解释

15号给出的CFM使用建议必须原样带入16–18号消融：

- 真实谱从头训练；
- 真实无标签预训练后微调；
- 真实谱＋同化合物CFM增强；
- 真实谱＋新增标签化合物CFM；
- CFM预训练、最终只用真实谱微调；
- CFM-only训练、真实谱测试负面对照。

即使域差距很大，课题仍然有价值：这会证明“合成谱不能简单替代实测谱”，并指出CFM真正适合的训练位置。
ChemVista必须继续锁定到模型、权重、阈值和适用域规则全部冻结之后。